# 03 - Build Silver + Gold
**Silver:** light typing (dates/booleans) and pass-through of the eight bronze tables.
**Gold:** the cross-system model. `gold.project_schedule_risk` fuses **non-SAP** schedule
signals with **SAP** cost + procurement signals into one governed, per-project risk table —
the single load-bearing join no source system does alone. Curated driver tables support drill-down.


In [ ]:
from pyspark.sql import functions as F

# ---- SILVER: typed pass-through ----
spark.sql('CREATE SCHEMA IF NOT EXISTS silver')

date_cols = {
    'dim_project': ['start_date', 'planned_finish', 'forecast_finish'],
    'fact_schedule_activity': ['baseline_start', 'baseline_finish', 'forecast_finish', 'actual_finish'],
    'sap_fi_cost': ['period'],
    'sap_mm_po': ['promised_date', 'revised_date'],
    'fact_engineering_change': ['issued_date'],
    'ext_disruption_signal': ['event_date'],
}
tables = ['dim_project', 'dim_wbs', 'fact_schedule_activity', 'sap_fi_cost',
          'sap_mm_po', 'sap_supplier', 'fact_engineering_change', 'ext_disruption_signal']
for t in tables:
    df = spark.table(f'bronze.{t}')
    for c in date_cols.get(t, []):
        if c in df.columns:
            df = df.withColumn(c, F.to_date(F.col(c)))
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'silver.{t}')
    print(f'silver.{t} ok')


In [ ]:
# ---- GOLD: create the schema (driver aggregations are inlined as CTEs below) ----
spark.sql('CREATE SCHEMA IF NOT EXISTS gold')
print('gold schema ready')


In [ ]:
# ---- GOLD: the fused per-project schedule-risk table (single statement) ----
# All driver aggregations are inlined as CTEs that read silver.* DIRECTLY. Temp views were
# tried first but a CTAS on a schema-enabled Lakehouse re-resolves a temp view's body in a
# different catalog context and fails with TABLE_OR_VIEW_NOT_FOUND on silver.*; direct
# silver.* references inside the CTAS resolve fine. Also avoids reading the gold table while
# overwriting it (self-referential CREATE OR REPLACE fails on Delta).
spark.sql('''
CREATE OR REPLACE TABLE gold.project_schedule_risk
USING delta AS
WITH v_sched AS (            -- non-SAP schedule signals (Primavera)
    SELECT project_id,
           MAX(GREATEST(DATEDIFF(forecast_finish, baseline_finish), 0)) AS max_slip_days,
           MIN(total_float_days) AS min_float,
           SUM(CASE WHEN is_critical_path AND forecast_finish > baseline_finish THEN 1 ELSE 0 END) AS cp_at_risk
    FROM silver.fact_schedule_activity GROUP BY project_id
),
v_po AS (                   -- SAP procurement signal (late long-lead POs)
    SELECT project_id,
           SUM(CASE WHEN is_long_lead AND status = 'Late' THEN 1 ELSE 0 END) AS late_long_lead_pos
    FROM silver.sap_mm_po GROUP BY project_id
),
v_cost AS (                 -- SAP finance signal (overrun + cost-to-complete + EV)
    SELECT project_id,
           SUM(forecast_cost - budget) AS forecast_overrun,
           SUM(cost_to_complete)       AS cost_to_complete,
           SUM(earned_value)           AS earned_value
    FROM silver.sap_fi_cost GROUP BY project_id
),
scored AS (
    SELECT
        p.project_id, p.project_name, p.client, p.region, p.contract_type,
        p.pct_complete, p.planned_finish, p.forecast_finish,
        COALESCE(s.max_slip_days, 0)       AS schedule_slip_days,      -- non-SAP
        COALESCE(s.min_float, 0)           AS min_total_float_days,    -- non-SAP
        COALESCE(s.cp_at_risk, 0)          AS critical_path_at_risk,   -- non-SAP
        COALESCE(po.late_long_lead_pos, 0) AS late_long_lead_pos,      -- SAP
        COALESCE(c.forecast_overrun, 0)    AS forecast_overrun,        -- SAP
        COALESCE(c.cost_to_complete, 0)    AS cost_to_complete,        -- SAP
        COALESCE(c.earned_value, 0)        AS earned_value,            -- SAP
        LEAST(100,
            COALESCE(s.max_slip_days, 0) * 1.5
          + CASE WHEN COALESCE(s.min_float, 0) < 0 THEN -s.min_float ELSE 0 END * 2
          + COALESCE(s.cp_at_risk, 0) * 3
          + COALESCE(po.late_long_lead_pos, 0) * 5
          + COALESCE(c.forecast_overrun, 0) / 100000
        ) AS schedule_risk_score
    FROM silver.dim_project p
    LEFT JOIN v_sched s ON p.project_id = s.project_id
    LEFT JOIN v_po   po ON p.project_id = po.project_id
    LEFT JOIN v_cost c  ON p.project_id = c.project_id
)
SELECT *,
    CASE WHEN schedule_risk_score >= 61 THEN 'Red'
         WHEN schedule_risk_score >= 26 THEN 'Amber'
         ELSE 'Green' END AS risk_band
FROM scored
''')

spark.sql('''
ALTER TABLE gold.project_schedule_risk SET TBLPROPERTIES ('note' = 'fuses SAP + non-SAP signals')
''')

print('gold.project_schedule_risk built. Top 5 by risk:')
spark.sql('''
SELECT project_id, project_name, ROUND(schedule_risk_score,1) AS score, risk_band,
       schedule_slip_days, min_total_float_days, critical_path_at_risk,
       late_long_lead_pos, ROUND(forecast_overrun,0) AS overrun
FROM gold.project_schedule_risk ORDER BY schedule_risk_score DESC LIMIT 5
''').show(truncate=False)


In [ ]:
# ---- GOLD: curated driver-detail tables for dashboard + agent drill-down ----
# Falcon's cross-system 'why' on one row set: at-risk critical-path activities + their EC (non-SAP),
# the late long-lead PO + supplier (SAP), and the cost exposure (SAP).
spark.sql('''
CREATE OR REPLACE TABLE gold.at_risk_activities USING delta AS
SELECT a.project_id, a.wbs_id, a.activity_id, a.activity_name, a.discipline_hint,
       a.baseline_finish, a.forecast_finish, a.total_float_days, a.is_critical_path,
       e.ec_id, e.title AS ec_title, e.status AS ec_status, e.schedule_impact_days
FROM (SELECT fsa.*, w.discipline AS discipline_hint
      FROM silver.fact_schedule_activity fsa
      JOIN silver.dim_wbs w ON fsa.wbs_id = w.wbs_id) a
LEFT JOIN silver.fact_engineering_change e ON e.affected_activity_id = a.activity_id
WHERE a.is_critical_path AND a.forecast_finish > a.baseline_finish
''')

spark.sql('''
CREATE OR REPLACE TABLE gold.late_procurement USING delta AS
SELECT po.project_id, po.wbs_id, po.po_id, po.material_desc, po.is_long_lead,
       po.promised_date, po.revised_date, DATEDIFF(po.revised_date, po.promised_date) AS days_late,
       s.supplier_id, s.supplier_name, s.country, s.risk_rating
FROM silver.sap_mm_po po
JOIN silver.sap_supplier s ON po.supplier_id = s.supplier_id
WHERE po.status = 'Late'
''')
print('gold.at_risk_activities and gold.late_procurement built.')
print('Gold layer complete.')
